<a href="https://colab.research.google.com/github/333universe/NPKmathClaud/blob/main/nkp_run_allx1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NKP-Math + Sieve + Ledger + Real-claims test: clean run order (v3)
Run the cells **top to bottom**. Cells 1-8 only write files. Cells 9-17 run checks. Part G is the real-claims test.

**Rule:** after any AI edit, run cell 10 (the gate). `RESULT: ALL PASS` = safe to continue. Empty output or any FAIL = stop.

Needs only sympy, mpmath, numpy (preinstalled in Colab) and sqlite3 (part of Python).

## Part A - write the files (cells 1-8)

In [1]:
%%writefile nkp_math.py
"""
NKP-Math: claim checking where claims are MATHEMATICAL OBJECTS and sources are COMPUTATIONS.

Text engines can only compare strings; this one asks "is it true?" and computes the answer.
Claim types (each returns a Verdict):
  identity(lhs, rhs)            exact algebra, else 120-digit testing at positive AND negative points (no domain stated = all reals)
  value(expr, "3.14159")        evaluate to 60 digits, compare to the claimed decimal at ITS precision
  exceeds(expr, x, lo, hi, c)   does |f(x)| exceed c somewhere?  interval arithmetic + branch-and-bound:
                                can PROVE "never" (sup bound) or FIND a witness
  dimensions("F = m*a", dims)   units check, with symbolic exponents ("consistent only if b = 0")
  coincidence(a, b, formula)    evidence in BITS = precision bits - formula bits - search bits
Statuses: PROVED, VERIFIED, SUPPORTED, CONSISTENT (accept) | REFUTED, ILL_FORMED, COINCIDENCE (reject)
          CONDITIONAL, UNDETERMINED, SIGNIFICANT (reported as-is)
A SourceLedger keeps the track record of whoever made each claim (NKP's source-reliability idea, but
with ground truth supplied by computation instead of consensus).

Needs sympy and mpmath (both preinstalled in Colab).   python nkp_math.py --test
"""
from __future__ import annotations

import math
import random
import re
import sys
import time
from collections import Counter
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import mpmath as mp
import sympy as sp

mp.mp.dps = 50
ACCEPT = {"PROVED", "VERIFIED", "SUPPORTED", "CONSISTENT", "SIGNIFICANT"}
REJECT = {"REFUTED", "ILL_FORMED", "COINCIDENCE"}

_FUNCS = ("sin", "cos", "tan", "exp", "log", "sqrt", "Abs", "atan", "sinh", "cosh", "tanh")
_GLOBALS = {k: getattr(sp, k) for k in ("Symbol", "Integer", "Float", "Rational", "Function", "Dummy") + _FUNCS}
_GLOBALS["__builtins__"] = {}
_LOCALS = {"phi": sp.GoldenRatio, "pi": sp.pi, "euler": sp.E}
_TRANSFORMS = sp.parsing.sympy_parser.standard_transformations + (sp.parsing.sympy_parser.convert_xor,)


def parse(s: str):
    """Parse text into a sympy expression.  Decimals become exact Rationals ("0.3333" is NOT a binary float)."""
    s = re.sub(r"(?<![\w.])(\d+\.\d+)(?![\w.])", r"Rational('\1')", s)
    return sp.parse_expr(s, local_dict=dict(_LOCALS), global_dict=dict(_GLOBALS), transformations=_TRANSFORMS)


@dataclass
class Verdict:
    status: str
    detail: str
    ms: float = 0.0

    @property
    def accepted(self) -> bool:
        return self.status in ACCEPT


class SourceLedger:
    """Track record of claim sources, scored by what computation found."""

    def __init__(self):
        self.c: Dict[str, Counter] = {}

    def record(self, source: Optional[str], status: str) -> None:
        if source is None:
            return
        k = self.c.setdefault(source, Counter())
        if status in ACCEPT:
            k["true"] += 1
        elif status in REJECT:
            k["false"] += 1
        else:
            k["open"] += 1

    def reliability(self, source: str) -> float:
        k = self.c.get(source, Counter())
        return (k["true"] + 1) / (k["true"] + k["false"] + 2)      # Beta(1,1) prior

    def table(self) -> List[Tuple[str, int, int, int, float]]:
        return sorted(((s, k["true"], k["false"], k["open"], self.reliability(s)) for s, k in self.c.items()),
                      key=lambda r: -r[4])


class NKPMath:
    def __init__(self, n_points: int = 12, seed: int = 0, node_limit: int = 4000):
        self.n_points, self.rng, self.node_limit = n_points, random.Random(seed), node_limit
        self.ledger = SourceLedger()

    def _done(self, v: Verdict, t0: float, source: Optional[str]) -> Verdict:
        v.ms = (time.perf_counter() - t0) * 1000
        self.ledger.record(source, v.status)
        return v

    # ------------------------------------------------------------------ identity
    @staticmethod
    def _rational_nonzero(expr, syms) -> bool:
        """True only if expr is PROVABLY a nonzero rational function with rational coefficients."""
        try:
            num, _ = sp.fraction(sp.cancel(expr))
            P = sp.Poly(num, *syms)
            return (not P.is_zero) and all(c.is_Rational for c in P.coeffs())
        except Exception:
            return False

    def identity(self, lhs: str, rhs: str, domain: Optional[Dict[str, Tuple[float, float]]] = None,
                 source: Optional[str] = None) -> Verdict:
        """Is lhs == rhs?  Without a declared domain the claim is read as 'for all real inputs': it is tested on
        positive AND negative values, and a claim true only for positives is reported CONDITIONAL."""
        t0 = time.perf_counter()
        L, R = parse(lhs), parse(rhs)
        d = L - R
        syms = sorted(d.free_symbols, key=str)
        if not syms:
            if sp.simplify(d) == 0:
                return self._done(Verdict("PROVED", "exact: the difference simplifies to 0"), t0, source)
            val = sp.N(d, 60)
            if abs(val) > sp.Float(10) ** -40:
                return self._done(Verdict("REFUTED", f"lhs - rhs = {sp.N(d, 12)}"), t0, source)
            return self._done(Verdict("SUPPORTED", "agrees to 40 digits (not proved symbolically)"), t0, source)
        # exact algebra first: catches differences of ANY size between rational expressions
        try:
            dc = sp.cancel(d)
        except Exception:
            dc = d
        if dc == 0:
            return self._done(Verdict("PROVED", "exact: the rational difference cancels to 0"), t0, source)
        if self._rational_nonzero(d, syms):
            return self._done(Verdict("REFUTED", f"exact: lhs - rhs = {sp.sstr(dc)[:70]}, provably nonzero"), t0, source)
        DIGITS, TOL = 120, mp.mpf(10) ** -90
        with mp.workdps(DIGITS):
            fL, fR = sp.lambdify(syms, L, modules="mpmath"), sp.lambdify(syms, R, modules="mpmath")

            def first_failure(dom, signed):
                n_pts = max(self.n_points, 2 ** len(syms)) if signed else self.n_points
                for i in range(n_pts):
                    pt = []
                    for k, s_ in enumerate(syms):
                        lo, hi = dom.get(str(s_), (0.1, 3.0))
                        v = mp.mpf(lo) + (mp.mpf(hi) - mp.mpf(lo)) * mp.mpf(self.rng.getrandbits(420)) / mp.mpf(2) ** 420
                        neg = ((i >> k) & 1) if i < 2 ** len(syms) else (self.rng.random() < 0.5)   # every sign pattern is tried
                        pt.append(-v if (signed and neg) else v)
                    a_, b_ = fL(*pt), fR(*pt)
                    if abs(a_ - b_) > TOL * max(1, abs(a_), abs(b_)):
                        where = ", ".join(f"{s_}={mp.nstr(v, 6)}" for s_, v in zip(syms, pt))
                        return f"{where}: lhs-rhs = {mp.nstr(a_ - b_, 6)}"
                return None

            fail = first_failure(domain or {}, signed=False)
            if fail:
                return self._done(Verdict("REFUTED", f"counterexample at {fail}"), t0, source)
            if domain is None:
                fail = first_failure({}, signed=True)
                if fail:
                    return self._done(Verdict("CONDITIONAL", f"holds for positive inputs only; fails at {fail}"), t0, source)
        if sp.simplify(d) == 0 or sp.simplify(sp.expand_trig(d)) == 0:
            return self._done(Verdict("PROVED", "symbolic: the difference simplifies to 0"), t0, source)
        return self._done(Verdict("SUPPORTED", f"agrees to 90 digits at {self.n_points} random points (positive and negative); not proved"), t0, source)

    # ------------------------------------------------------------------ value
    def value(self, expr: str, claimed: str, source: Optional[str] = None) -> Verdict:
        t0 = time.perf_counter()
        decimals = len(claimed.split(".")[1]) if "." in claimed else 0
        with mp.workdps(max(50, decimals + 30)):            # compute well beyond the digits being claimed
            v = mp.mpmathify(str(sp.N(parse(expr), decimals + 40)))
            tol = mp.mpf(10) ** (-decimals) / 2
            err = abs(v - mp.mpf(claimed))
            if err <= tol * (1 + mp.mpf(10) ** -15):
                return self._done(Verdict("VERIFIED", f"matches to the claimed {decimals} decimals (true value {mp.nstr(v, min(decimals + 3, 40))})"), t0, source)
            return self._done(Verdict("REFUTED", f"claimed {claimed[:50]}; true value {mp.nstr(v, min(decimals + 3, 40))} (error {mp.nstr(err, 3)})"), t0, source)

    # ------------------------------------------------------------------ bound (interval arithmetic)
    def exceeds(self, expr: str, var: str, lo, hi, c, source: Optional[str] = None) -> Verdict:
        """Claim: there is an x in [lo, hi] with |f(x)| > c.  Proves 'never' or finds a witness."""
        t0 = time.perf_counter()
        x = sp.Symbol(var)
        e = parse(expr).subs(sp.GoldenRatio, (1 + sp.sqrt(5)) / 2)
        iv = mp.iv
        ns = {n: getattr(iv, n) for n in ("sin", "cos", "tan", "exp", "log", "sqrt") if hasattr(iv, n)}
        ns.update({"Abs": abs, "pi": iv.pi, "e": iv.e, "mpf": iv.mpf})   # constants must be intervals too          # (mpmath intervals have no atan/sinh: such terms are unsupported)
        f_iv = sp.lambdify(x, e, modules=[ns, "mpmath"])
        f_pt = sp.lambdify(x, e, modules="mpmath")
        c = mp.mpf(str(c)) if isinstance(c, str) else mp.mpf(c)
        lo, hi = mp.mpf(lo), mp.mpf(hi)

        def upper(a, b):
            r = f_iv(iv.mpf([a, b]))
            return max(abs(r.a), abs(r.b)) if hasattr(r, "a") else abs(r)

        U = upper(lo, hi)
        if U <= c:
            return self._done(Verdict("REFUTED", f"proved by interval arithmetic: |f| <= {mp.nstr(U, 8)} <= {mp.nstr(c, 8)} on [{lo}, {hi}]"), t0, source)
        stack, nodes, width0 = [(lo, hi)], 0, hi - lo
        while stack and nodes < self.node_limit:
            a, b = stack.pop()
            nodes += 1
            if upper(a, b) <= c:
                continue
            m = (a + b) / 2
            fm = abs(f_pt(m))
            if fm > c:
                return self._done(Verdict("PROVED", f"witness x = {mp.nstr(m, 12)}: |f| = {mp.nstr(fm, 10)} > {mp.nstr(c, 8)}"), t0, source)
            if (b - a) < width0 * mp.mpf(10) ** -14:
                continue
            stack.extend([(a, m), (m, b)])
        if not stack:
            return self._done(Verdict("REFUTED", f"proved by branch-and-bound: |f| <= {mp.nstr(c, 8)} everywhere ({nodes} boxes)"), t0, source)
        return self._done(Verdict("UNDETERMINED", f"no witness and no proof after {nodes} boxes"), t0, source)

    # ------------------------------------------------------------------ dimensions
    def dimensions(self, equation: str, dims: Dict[str, object], source: Optional[str] = None) -> Verdict:
        t0 = time.perf_counter()
        D = {k: self._dimspec(v) for k, v in dims.items()}
        issues: List[str] = []
        constraints: List[sp.Expr] = []

        def sub(a, b):
            return {k: sp.simplify(a.get(k, 0) - b.get(k, 0)) for k in set(a) | set(b)}

        def same(a, b, ctx):
            for k, dv in sub(a, b).items():
                if dv == 0:
                    continue
                if dv.free_symbols:
                    constraints.append(dv)
                else:
                    issues.append(f"{ctx}: [{self._fmt(a)}] vs [{self._fmt(b)}]")
                    return

        def dim(e):
            if e.is_Number or e in (sp.pi, sp.E, sp.GoldenRatio):
                return {}
            if e.is_Symbol:
                return dict(D.get(str(e), {}))
            if e.is_Add:
                first = dim(e.args[0])
                for a in e.args[1:]:
                    same(first, dim(a), f"cannot add terms in {sp.sstr(e)}")
                return first
            if e.is_Mul:
                out: Dict[str, sp.Expr] = {}
                for a in e.args:
                    for k, v in dim(a).items():
                        out[k] = sp.simplify(out.get(k, 0) + v)
                return out
            if e.is_Pow:
                b, ex = e.args
                if dim(ex):
                    issues.append(f"exponent has units in {sp.sstr(e)}")
                bd = dim(b)
                return {k: sp.simplify(v * ex) for k, v in bd.items()} if bd else {}
            if e.func in (sp.sin, sp.cos, sp.tan, sp.exp, sp.log, sp.atan, sp.sinh, sp.cosh, sp.tanh):
                same({}, dim(e.args[0]), f"argument of {e.func.__name__} must be dimensionless, in {sp.sstr(e)}")
                return {}
            if e.func == sp.Abs:
                return dim(e.args[0])
            issues.append(f"unsupported term {sp.sstr(e)}")
            return {}

        lhs, rhs = equation.split("=")
        same(dim(parse(lhs)), dim(parse(rhs)), "both sides of the equation")
        if issues:
            return self._done(Verdict("ILL_FORMED", "; ".join(issues)), t0, source)
        if constraints:
            unknowns = sorted({s for c in constraints for s in c.free_symbols}, key=str)
            sol = sp.solve(constraints, unknowns, dict=True)
            if not sol:
                return self._done(Verdict("ILL_FORMED", f"no choice of {unknowns} makes the units agree"), t0, source)
            return self._done(Verdict("CONDITIONAL", f"units agree ONLY if {sol[0]}"), t0, source)
        return self._done(Verdict("CONSISTENT", "units agree"), t0, source)

    @staticmethod
    def _dimspec(v):
        if isinstance(v, dict):
            return {k: sp.nsimplify(x) for k, x in v.items()}
        out = {}
        for tok in str(v).split():
            m = re.fullmatch(r"([A-Za-z]+)(?:\^(-?[\d/]+))?", tok)
            out[m.group(1)] = sp.Rational(m.group(2)) if m.group(2) else sp.Integer(1)
        return out

    @staticmethod
    def _fmt(d):
        s = " ".join(f"{k}^{v}" for k, v in sorted(d.items()) if v != 0)
        return s or "dimensionless"

    # ------------------------------------------------------------------ coincidence meter
    @staticmethod
    def formula_bits(formula: str, n_leaf: int = 30, n_op: int = 6) -> float:
        e = parse(formula)
        leaves = [a for a in sp.preorder_traversal(e) if a.is_Atom]
        bits = 0.0
        for a in leaves:
            bits += math.log2(n_leaf)
            if a.is_Integer and abs(a) > 12:
                bits += math.log2(abs(int(a)) + 1)          # big integers cost more to specify
        return bits + sp.count_ops(e) * math.log2(n_op)

    def coincidence(self, a: float, b: float, formula: str = "", rel_unc: float = 0.0,
                    search_bits: float = 8.6, threshold: float = 10.0, source: Optional[str] = None) -> Verdict:
        """Is 'a is close to b' evidence of a real relation?  net bits = precision - formula - search."""
        t0 = time.perf_counter()
        rel = max(abs(a - b) / max(abs(a), abs(b)), rel_unc)
        prec = math.log2(1 / (2 * rel)) if rel > 0 else 64.0
        fb = self.formula_bits(formula) if formula else 0.0
        net = prec - fb - search_bits
        txt = f"match to {rel:.2e} = {prec:.1f} bits; formula costs {fb:.1f} bits; search freedom {search_bits:.1f} bits; net {net:+.1f} bits"
        return self._done(Verdict("SIGNIFICANT" if net >= threshold else "COINCIDENCE", txt), t0, source)


# ----------------------------------------------------------------------------- tests
def run_tests() -> None:
    m = NKPMath()
    assert m.identity("sin(x)**2 + cos(x)**2", "1").status == "PROVED"
    r = m.identity("sin(x)", "x - x**3/6")
    assert r.status == "REFUTED" and "counterexample" in r.detail
    assert m.identity("x", "x + 10**(-20)*x**2").status == "REFUTED"                    # a 1e-20 near-identity
    assert m.identity("3*0.3333", "1").status == "REFUTED" # exact fraction checking
    assert m.value("pi", "3.14159").status == "VERIFIED"
    assert m.value("pi", "3.14150").status == "REFUTED"
    assert m.exceeds("sin(x)", "x", 0.0, 7.0, 1.1).status == "REFUTED"
    assert m.exceeds("x*(2-x)", "x", 0.0, 2.0, 0.99).status == "PROVED"
    assert m.dimensions("F = m*a", {"F": "M L T^-2", "m": "M", "a": "L T^-2"}).status == "CONSISTENT"
    assert m.dimensions("v = u + a*t^b", {"v": "L T^-1", "u": "L T^-1", "a": "L T^-2", "t": "T"}).status == "CONDITIONAL"
    assert m.dimensions("v = u + a*t^2", {"v": "L T^-1", "u": "L T^-1", "a": "L T^-2", "t": "T"}).status == "ILL_FORMED"
    assert m.coincidence(1.0000001, 1.0, "1", search_bits=2.0, threshold=10.0).status == "SIGNIFICANT"
    assert m.coincidence(1.01, 1.0, "1", search_bits=2.0, threshold=10.0).status == "COINCIDENCE"
    print("nkp_math.py tests passed")

if __name__ == "__main__":
    if "--test" in sys.argv:
        run_tests()


Writing nkp_math.py


In [2]:
!python nkp_math.py --test

nkp_math.py tests passed


In [3]:
%%writefile sieve.py
"""
Sieve: A basic prime sieve and utility library to support our mathematical ledger operations.
"""
import math

def prime_sieve(limit: int):
    """Return a list of primes up to limit using the Sieve of Eratosthenes."""
    if limit < 2:
        return []
    sieve = [True] * (limit + 1)
    sieve[0] = sieve[1] = False
    for i in range(2, int(math.isqrt(limit)) + 1):
        if sieve[i]:
            for j in range(i*i, limit + 1, i):
                sieve[j] = False
    return [i for i, is_prime in enumerate(sieve) if is_prime]

if __name__ == '__main__':
    primes = prime_sieve(30)
    print(f"Primes up to 30: {primes}")
    assert primes == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]


Writing sieve.py


In [4]:
!python sieve.py

Primes up to 30: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]


In [5]:
%%writefile ledger.py
"""
Ledger: Simple block/transaction cryptographic ledger simulation for mathematical assertion histories.
"""
import hashlib
import json
import time
from typing import Any, Dict, List

class Block:
    def __init__(self, index: int, transactions: List[Dict[str, Any]], previous_hash: str):
        self.index = index
        self.timestamp = time.time()
        self.transactions = transactions
        self.previous_hash = previous_hash
        self.nonce = 0
        self.hash = self.compute_hash()

    def compute_hash(self) -> str:
        block_string = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "transactions": self.transactions,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }, sort_keys=True)
        return hashlib.sha256(block_string.encode()).hexdigest()

    def mine(self, difficulty: int) -> str:
        target = "0" * difficulty
        while not self.hash.startswith(target):
            self.nonce += 1
            self.hash = self.compute_hash()
        return self.hash

class Ledger:
    def __init__(self, difficulty: int = 2):
        self.chain: List[Block] = []
        self.difficulty = difficulty
        self.pending_transactions: List[Dict[str, Any]] = []
        self.create_genesis_block()

    def create_genesis_block(self):
        genesis_block = Block(0, [{"info": "Genesis Block"}], "0")
        genesis_block.mine(self.difficulty)
        self.chain.append(genesis_block)

    def add_transaction(self, sender: str, recipient: str, claim: Dict[str, Any], verdict: str):
        self.pending_transactions.append({
            "sender": sender,
            "recipient": recipient,
            "claim": claim,
            "verdict": verdict,
            "time": time.time()
        })

    def mine_pending(self, miner_address: str):
        if not self.pending_transactions:
            return False
        last_block = self.chain[-1]
        new_block = Block(
            index=last_block.index + 1,
            transactions=self.pending_transactions,
            previous_hash=last_block.hash
        )
        new_block.mine(self.difficulty)
        self.chain.append(new_block)
        self.pending_transactions = [{"reward_to": miner_address, "amount": 1.0}]
        return True

if __name__ == '__main__':
    ledger = Ledger(difficulty=2)
    ledger.add_transaction("Alice", "System", {"identity": "sin(x)^2 + cos(x)^2 = 1"}, "PROVED")
    success = ledger.mine_pending("Miner1")
    assert success
    assert len(ledger.chain) == 2
    print(f"Ledger generated. Blocks count: {len(ledger.chain)}")
    print(f"Latest block hash: {ledger.chain[-1].hash}")


Writing ledger.py


In [6]:
!python ledger.py

Ledger generated. Blocks count: 2
Latest block hash: 00c77bc67dbf2639742018556afe1713dc16e6399a17403ebc4ce77246112598


In [7]:
%%writefile ledger.py
"""
Ledger: Simple block/transaction cryptographic ledger simulation for mathematical assertion histories.
"""
import hashlib
import json
import time
from typing import Any, Dict, List

class Block:
    def __init__(self, index: int, transactions: List[Dict[str, Any]], previous_hash: str):
        self.index = index
        self.timestamp = time.time()
        self.transactions = transactions
        self.previous_hash = previous_hash
        self.nonce = 0
        self.hash = self.compute_hash()

    def compute_hash(self) -> str:
        block_string = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "transactions": self.transactions,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }, sort_keys=True)
        return hashlib.sha256(block_string.encode()).hexdigest()

    def mine(self, difficulty: int) -> str:
        target = "0" * difficulty
        while not self.hash.startswith(target):
            self.nonce += 1
            self.hash = self.compute_hash()
        return self.hash

class Ledger:
    def __init__(self, difficulty: int = 2):
        self.chain: List[Block] = []
        self.difficulty = difficulty
        self.pending_transactions: List[Dict[str, Any]] = []
        self.create_genesis_block()

    def create_genesis_block(self):
        genesis_block = Block(0, [{"info": "Genesis Block"}], "0")
        genesis_block.mine(self.difficulty)
        self.chain.append(genesis_block)

    def add_transaction(self, sender: str, recipient: str, claim: Dict[str, Any], verdict: str):
        self.pending_transactions.append({
            "sender": sender,
            "recipient": recipient,
            "claim": claim,
            "verdict": verdict,
            "time": time.time()
        })

    def mine_pending(self, miner_address: str):
        if not self.pending_transactions:
            return False
        last_block = self.chain[-1]
        new_block = Block(
            index=last_block.index + 1,
            transactions=self.pending_transactions,
            previous_hash=last_block.hash
        )
        new_block.mine(self.difficulty)
        self.chain.append(new_block)
        self.pending_transactions = [{"reward_to": miner_address, "amount": 1.0}]
        return True

if __name__ == '__main__':
    ledger = Ledger(difficulty=2)
    ledger.add_transaction("Alice", "System", {"identity": "sin(x)^2 + cos(x)^2 = 1"}, "PROVED")
    success = ledger.mine_pending("Miner1")
    assert success
    assert len(ledger.chain) == 2
    print(f"Ledger generated. Blocks count: {len(ledger.chain)}")
    print(f"Latest block hash: {ledger.chain[-1].hash}")


Overwriting ledger.py


In [8]:
!python ledger.py

Ledger generated. Blocks count: 2
Latest block hash: 005b3f19038426cd4f3a06777a87ce5b0251ad24b1810045edaf85abab50aeac


In [9]:
%%writefile run_checks.py
"""
Run Checks: Validates assertions through our NKPMath engine and logs transactions inside the secure cryptographic ledger.
"""
import sys
from nkp_math import NKPMath
from ledger import Ledger
from sieve import prime_sieve

def run_suite():
    print("--- Executing Mathematical Claim Verification Gate ---")
    math_engine = NKPMath()
    ledger_engine = Ledger(difficulty=2)

    # Test algebraic identity
    v1 = math_engine.identity("sin(x)**2 + cos(x)**2", "1", source="IdentityChecker")
    print(f"Identity sin^2(x) + cos^2(x) = 1: {v1.status} ({v1.detail})")
    ledger_engine.add_transaction("IdentityChecker", "Ledger", {"lhs": "sin(x)**2+cos(x)**2", "rhs": "1"}, v1.status)

    # Test numerical verification
    v2 = math_engine.value("pi", "3.141592653589793", source="ValueChecker")
    print(f"Value verification of Pi: {v2.status} ({v2.detail})")
    ledger_engine.add_transaction("ValueChecker", "Ledger", {"expr": "pi", "val": "3.141592653589793"}, v2.status)

    # Check prime distribution claims (Sieve component validation)
    primes_under_50 = prime_sieve(50)
    primes_count = len(primes_under_50)
    expected_count = 15  # 15 primes below 50
    sieve_status = "VERIFIED" if primes_count == expected_count else "REFUTED"
    print(f"Primes under 50 count claim: {sieve_status} (Found {primes_count}, expected {expected_count})")
    ledger_engine.add_transaction("SieveChecker", "Ledger", {"limit": 50, "expected": expected_count}, sieve_status)

    # Execute Block Mining sequence on pending operations
    success = ledger_engine.mine_pending("Validation_Miner_Node")
    if success and v1.accepted and v2.accepted and sieve_status == "VERIFIED":
        print("\nRESULT: ALL PASS")
    else:
        print("\nRESULT: FAIL - Validation errors identified")
        sys.exit(1)

if __name__ == '__main__':
    run_suite()


Writing run_checks.py


In [10]:
!python run_checks.py

--- Executing Mathematical Claim Verification Gate ---
Identity sin^2(x) + cos^2(x) = 1: PROVED (symbolic: the difference simplifies to 0)
Value verification of Pi: VERIFIED (matches to the claimed 15 decimals (true value 3.14159265358979324))
Primes under 50 count claim: VERIFIED (Found 15, expected 15)

RESULT: ALL PASS


!python real_claims_test.py

## Part G - Real-Claims Test
Now we proceed to test real mathematical claims (such as inequalities, limits, or coincidences) by feeding them into our verified NKP-Math engine.

In [12]:
%%writefile real_claims_test.py
"""
Part G: Real-Claims Test
Tests non-trivial mathematical claims and logs them inside the mathematical ledger.
"""
from nkp_math import NKPMath
from ledger import Ledger

def run_real_claims():
    print("--- Running Part G: Real-Claims Test ---")
    math_engine = NKPMath()
    ledger_engine = Ledger(difficulty=2)

    # Claim 1: Ramanujan's constant near-integer coincidence: e^(pi * sqrt(163))
    # Expected near-integer: 262537412640768743.99999999999925...
    ramanujan_expr = "exp(pi * sqrt(163))"
    v1 = math_engine.value(ramanujan_expr, "262537412640768744", source="RamanujanCoincidence")
    print(f"Ramanujan Constant Coincidence: {v1.status} ({v1.detail})")
    ledger_engine.add_transaction("RamanujanCoincidence", "Ledger", {"expr": ramanujan_expr}, v1.status)

    # Claim 2: Trigonometric Identity cos(x)^2 - sin(x)^2 = cos(2*x)
    v2 = math_engine.identity("cos(x)**2 - sin(x)**2", "cos(2*x)", source="TrigIdentity")
    print(f"Identity cos(x)^2 - sin(x)^2 = cos(2*x): {v2.status} ({v2.detail})")
    ledger_engine.add_transaction("TrigIdentity", "Ledger", {"lhs": "cos(x)**2 - sin(x)**2", "rhs": "cos(2*x)"}, v2.status)

    # Mine block to secure ledger history
    ledger_engine.mine_pending("Real_Claims_Miner_Node")
    print("\nReal-Claims logged inside ledger successfully.")

if __name__ == '__main__':
    run_real_claims()

Overwriting real_claims_test.py


In [13]:
!python real_claims_test.py

--- Running Part G: Real-Claims Test ---
Ramanujan Constant Coincidence: VERIFIED (matches to the claimed 0 decimals (true value 2.63e+17))
Identity cos(x)^2 - sin(x)^2 = cos(2*x): PROVED (symbolic: the difference simplifies to 0)

Real-Claims logged inside ledger successfully.


In [14]:
import json
from ledger import Ledger
from nkp_math import NKPMath
from sieve import prime_sieve

# Re-run a validation pass to populate the ledger
print("=== RECONSTRUCTING LEDGER ENTRIES ===")
math_engine = NKPMath()
ledger_engine = Ledger(difficulty=2)

# Add Part A-F check transactions
v1 = math_engine.identity("sin(x)**2 + cos(x)**2", "1")
ledger_engine.add_transaction("IdentityChecker", "Ledger", {"lhs": "sin(x)**2+cos(x)**2", "rhs": "1"}, v1.status)

v2 = math_engine.value("pi", "3.141592653589793")
ledger_engine.add_transaction("ValueChecker", "Ledger", {"expr": "pi", "val": "3.141592653589793"}, v2.status)

primes_under_50 = prime_sieve(50)
sieve_status = "VERIFIED" if len(primes_under_50) == 15 else "REFUTED"
ledger_engine.add_transaction("SieveChecker", "Ledger", {"limit": 50, "expected": 15}, sieve_status)
ledger_engine.mine_pending("Validation_Miner_Node")

# Add Part G real-claims transactions
ramanujan_expr = "exp(pi * sqrt(163))"
v3 = math_engine.value(ramanujan_expr, "262537412640768744")
ledger_engine.add_transaction("RamanujanCoincidence", "Ledger", {"expr": ramanujan_expr}, v3.status)

v4 = math_engine.identity("cos(x)**2 - sin(x)**2", "cos(2*x)")
ledger_engine.add_transaction("TrigIdentity", "Ledger", {"lhs": "cos(x)**2 - sin(x)**2", "rhs": "cos(2*x)"}, v4.status)
ledger_engine.mine_pending("Real_Claims_Miner_Node")

# Display chain block by block
for block in ledger_engine.chain:
    print(f"\nBlock #{block.index} | Hash: {block.hash} | Prev: {block.previous_hash}")
    print(json.dumps(block.transactions, indent=2))

=== RECONSTRUCTING LEDGER ENTRIES ===

Block #0 | Hash: 00f74661666beb8f930a8921ede1a3d9db01a54e27426b51dc82b4e77c1da58c | Prev: 0
[
  {
    "info": "Genesis Block"
  }
]

Block #1 | Hash: 00a31b7d2f2b20930f0eb51b7fb526a53a8c4b2d9febe124b74874f29a295712 | Prev: 00f74661666beb8f930a8921ede1a3d9db01a54e27426b51dc82b4e77c1da58c
[
  {
    "sender": "IdentityChecker",
    "recipient": "Ledger",
    "claim": {
      "lhs": "sin(x)**2+cos(x)**2",
      "rhs": "1"
    },
    "verdict": "PROVED",
    "time": 1791212668.2052608
  },
  {
    "sender": "ValueChecker",
    "recipient": "Ledger",
    "claim": {
      "expr": "pi",
      "val": "3.141592653589793"
    },
    "verdict": "VERIFIED",
    "time": 1791212668.205955
  },
  {
    "sender": "SieveChecker",
    "recipient": "Ledger",
    "claim": {
      "limit": 50,
      "expected": 15
    },
    "verdict": "VERIFIED",
    "time": 1791212668.206215
  }
]

Block #2 | Hash: 000ad01654c44ea75d7b92b9f85d308d242d7ac39a4fc5da0345dbde1c873d0a | Pr

### Verified Claims Summary Report
Below, we extract, analyze, and compile a structured report of all mathematical assertions recorded in the cryptographic blocks of the ledger.

In [15]:
import pandas as pd

report_data = []
for block in ledger_engine.chain:
    for tx in block.transactions:
        # Skip reward transactions
        if "sender" not in tx:
            continue

        sender = tx.get("sender")
        claim = tx.get("claim", {})
        verdict = tx.get("verdict")

        # Structure human-readable assertions
        if "lhs" in claim and "rhs" in claim:
            assertion = f"{claim['lhs']} == {claim['rhs']}"
        elif "expr" in claim:
            val_str = f" == {claim['val']}" if "val" in claim else " (Evaluation)"
            assertion = f"{claim['expr']}{val_str}"
        elif "limit" in claim:
            assertion = f"Primes count under {claim['limit']} is {claim['expected']}"
        else:
            assertion = str(claim)

        report_data.append({
            "Block #": block.index,
            "Claim Source": sender,
            "Assertion": assertion,
            "Verdict Status": verdict
        })

df_report = pd.DataFrame(report_data)
display(df_report)

,Block #,Claim Source,Assertion,Verdict Status
0,1,IdentityChecker,sin(x)**2+cos(x)**2 == 1,PROVED
1,1,ValueChecker,pi == 3.141592653589793,VERIFIED
2,1,SieveChecker,Primes count under 50 is 15,VERIFIED
3,2,RamanujanCoincidence,exp(pi * sqrt(163)) (Evaluation),VERIFIED
4,2,TrigIdentity,cos(x)**2 - sin(x)**2 == cos(2*x),PROVED
